# ESR rate based frequency sweep
This notebook contains an example of simulating a continous-wave ESR frequency sweep of a spin-1/2 inside an STM tunnel junction based on ESR rates. The ESR rate here competes with the relaxation rates of the tunneling electrons which leads to a different state occupations when at resonance. This change in occupation subsequently changes the tunneling current. The notebook is structered as:
1. Initializing a Spin-1/2 with Zeeman energies & Setting the tunnel parameters
2. Simulating the ESR spectrum based on the driving field $B_1$ and the total coherence time $T_2$
3. Comparing the simulated spectrum with the analytical solution of FWHM, amplitude and resonance frequency
4. Plot the ESR-driven populations

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const

from spinfinity import ESR, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf

%load_ext autoreload
%autoreload 2

## Initialize a single Spin-1/2
- Adding the Zeeman energy to the Hamiltonian'
- The tunnel rates determine relaxation rates which compete with the ESR driving
- tip-polarization and finite $V_\mathrm{DC}$ are important for the tunneling based readout

In [ ]:
spin = SpinSys([0.5])

# Hamiltonian paramters
spin.B = [0, 0, 0.5]
spin.gfactorvector = [2.0]

# Tunneling parameters
spin.T = 0.5 # K
spin.V_DC = 10.0  # mV
spin.TipPolarization = [0, 0, 0.5]
spin.U = np.array([0.0])
spin.G_st = 0.1e-9  # S
spin.G_ss = [1e-9]  # S
spin.G_tt = 1 # Allow for tip-tip scattering
spin.b0 = 0.0

# Initial Calculations
hamil.build_Hamiltonian(spin)
hamil.calc_EigEnergies(spin)
bath.calc_Rates(spin)
bath.calc_Populations(spin, AllowPumping=True)
I_bg = bath.calc_TunnelCurrent(spin, AllowPumping=True)
print(f"Background tunneling current: {I_bg*1e12:.3e} pA")

## Frequnecy Sweep

In [ ]:
# Parameters for the frequency sweep
T2 = 50.0  # ns; total coherence time which should fulfill T2 <= 2*T1
B1 = 2e-3  # T; microwave field amplitude 

freq_array = np.linspace(12, 16, 2000)  # GHz; frequency sweep range

# The calc function returns the frequency array, the current contrast, and the off-resonance current
_, DeltaI, I_off = ESR.calc_FreqSweep_ESRrate(
    spin,
    freq_array=freq_array,
    B1=B1,
    T2=T2,
    norm=False,
)

# The plot function returns a dictionary with the figure and data 
return_dict = ESR.plot_FreqSweep_ESRrate(spin, freq_array=freq_array, B1=B1, T2=T2, norm=False)


## Analytical Comparison
For the spin-1/2 one can compute the expected resonance paramters directly. With the angular Rabi frequency from the drive field: 
$$
\Omega = \gamma  B_1 m, \quad m= \left|\langle 0|\hat{S}_x|1\rangle\right|
$$
Where $m$ is the matrix element for a transition between the two states. The driving competes with the total coherence time $T_2$ and relaxation time $T_1$:
$$
T_1^{-1}=\Gamma_+ + \Gamma_-
$$
which is given by the sum of the inelastic excitation and relaxation rate. This leads to the FWHM:
$$
\mathrm{FWHM}
=
\frac{\sqrt{1+\Omega^2T_1T_2}}{\pi T_2}.
$$
And amplitude at resonance:
$$
A_{\mathrm{theory}}
=
\frac{(I_1-I_0)
\left(p_0^{\mathrm{off}}-p_1^{\mathrm{off}}\right)}{2}
\frac{\Omega^2T_1T_2}
{1+\Omega^2T_1T_2},
$$
Of the Lorentzian, having the form:
$$
\Delta I(f)
=
\frac{(I_1-I_0)
\left(p_0^{\mathrm{off}}-p_1^{\mathrm{off}}\right)}{2}
\frac{\Omega^2T_1T_2}
{1+4\pi^2(f-f_0)^2T_2^2+\Omega^2T_1T_2}.
$$
Here, the $I_0$ and $I_1$ is the tunneling current through the ground and excited state, respectively and $p_i^\mathrm{off}$ is the population of state $i$ off the resonance. 

In [ ]:
f_e = abs(spin.E_All_inGHz[1] - spin.E_All_inGHz[0])  # resonance frequency of the electron in GHz
p_off = spin.Populations.copy()  # populations of the two states without RF field

# Calculate the theoretical Rabi frequency and FWHM from the spin system parameters.
gamma = (
    spin.gfactorvector[0]
    * const.physical_constants["Bohr magneton"][0]
    / const.hbar
    * 1e-9
)  # rad/ns/T

# Matrix element of the Sx operator between the two states, in units of hbar
Sx_energy = spin.eigVectors.conj().T @ spin.Sx[0] @ spin.eigVectors
matrix_element = abs(Sx_energy[0, 1])

# Angualr Rabi frequency for the transition between the two states, in rad/ns
Omega = gamma * B1 * matrix_element                # rad/ns

# Both directions contribute to longitudinal relaxation.
Gamma_up = spin.Rates_Summed[0, 1]                # 1/s
Gamma_down = spin.Rates_Summed[1, 0]              # 1/s
T1 = 1e9 / (Gamma_up + Gamma_down)                # ns

assert T2 <= 2 * T1, "Total T2 must not exceed 2*T1."

s = Omega**2 * T1 * T2
FWHM_theory = np.sqrt(1 + s) / (np.pi * T2)       # GHz

# Common spin-independent background cancels from the current contrast.
state_currents = const.e * np.sum(
    spin.Rates[0] - spin.Rates[1], axis=1
)

A_theory = (
    (state_currents[1] - state_currents[0])
    * (p_off[0] - p_off[1])
    / 2
    * s
    / (1 + s)
)  # A

## Compare Simulation and Theory
The simulated spectrum is fitted to a Lorentzian and the fit paramters compared to the analytical solutions.

In [ ]:

# Fit the Fano function to the calculated current contrast data.
xfit, yfit, popt, perr = hf.fit_fano(freq_array, DeltaI)
A_fit, f0_fit, FWHM_fit, C_fit = popt


print(f"B1:                 {B1 * 1e3:.6f} mT")
print(f"T1 / T2:            {T1:.3f} / {T2:.3f} ns")
print(f"Resonance theory:   {f_e:.9f} GHz")
print(f"Resonance fit:      {f0_fit:.9f} GHz")
print(f"FWHM theory:        {FWHM_theory * 1e3:.6f} MHz")
print(f"FWHM fit:           {FWHM_fit * 1e3:.6f} MHz")
print(f"Amplitude theory:   {A_theory * 1e15:.6f} fA")
print(f"Amplitude fit:      {A_fit * 1e15:.6f} fA")

## Plotting the ESR-rate effect on the populations
To visualize how a specific driving does to the state populations, the following function shows a bar plot. For the used parameters, the populations equalize.

In [ ]:
return_dict = ESR.plot_ESRPopulations(spin, freq = f_e, B1 = B1, T2 = T2)